# imports

In [1]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# connection and load

In [2]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [3]:
ASSET = 'AAPL'
INTERVAL = '1h'

In [4]:
query = f"""
    SELECT * 
    FROM gold_ml_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

# dataframe

In [5]:
df = conn.execute(query).df()

# datetime index 

In [6]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

In [7]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,returns_5p,returns_10p,returns_20p,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low
date,,,,,,,,,,,,,,,,,,,,,
2024-05-13 16:30:00,AAPL,Stock,Yahoo Finance,1h,186.289993,186.929993,186.250000,186.658600,8098301.0,0.679993,...,0.020327,0.018851,0.021751,0.001977,0.003643,0.600889,186.289993,7602808.0,186.500000,185.779999
2024-05-13 17:30:00,AAPL,Stock,Yahoo Finance,1h,186.660004,186.949997,186.369995,186.650101,5453676.0,0.580002,...,0.019946,0.021677,0.020559,-0.000046,0.003107,0.482939,186.658600,8098301.0,186.929993,186.250000
2024-05-13 18:30:00,AAPL,Stock,Yahoo Finance,1h,186.659897,187.100006,186.559998,186.989899,6516486.0,0.540009,...,0.007006,0.024665,0.023257,0.001819,0.002888,0.796101,186.650101,5453676.0,186.949997,186.369995


# dropping cols and cleaning

In [8]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']

In [9]:
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 44


In [10]:
df.head(5)

,open,high,low,close,volume,daily_volatility,sma_7,sma_30,rsi_14,macd,...,returns_5p,returns_10p,returns_20p,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low
date,,,,,,,,,,,,,,,,,,,,,
2024-05-13 16:30:00,186.289993,186.929993,186.250000,186.658600,8098301.0,0.679993,184.748228,183.362917,73.207505,1.048488,...,0.020327,0.018851,0.021751,0.001977,0.003643,0.600889,186.289993,7602808.0,186.500000,185.779999
2024-05-13 17:30:00,186.660004,186.949997,186.369995,186.650101,5453676.0,0.580002,185.306813,183.533920,73.117205,1.135310,...,0.019946,0.021677,0.020559,-0.000046,0.003107,0.482939,186.658600,8098301.0,186.929993,186.250000
2024-05-13 18:30:00,186.659897,187.100006,186.559998,186.989899,6516486.0,0.540009,185.885369,183.681584,74.472907,1.217502,...,0.007006,0.024665,0.023257,0.001819,0.002888,0.796101,186.650101,5453676.0,186.949997,186.369995
2024-05-13 19:30:00,186.985001,187.029999,186.070007,186.285004,9250536.0,0.959991,186.354656,183.811884,66.932183,1.211792,...,0.001963,0.021327,0.015620,-0.003777,0.005153,0.223957,186.989899,6516486.0,187.100006,186.559998
2024-05-14 13:30:00,187.649994,188.300003,186.550003,187.009995,14754549.0,1.750000,186.543370,183.957717,70.266822,1.251342,...,0.003865,0.023366,0.018351,0.003884,0.009358,0.262852,186.285004,9250536.0,187.029999,186.070007


# to create target, in stocks we have overnight nad weekends so we need cannot use shift(-1)

In [11]:
df['target_1h'] = df['close'].pct_change().shift(-1)

# dropping the nan from target 

In [12]:
df = df.dropna(subset=['target_1h'])

In [13]:
df['target_direction'] = (df['target_1h'] > 0).astype(int)
balance = df['target_direction'].value_counts(normalize=True) * 100
print("Class Balance:")
print(balance)

Class Balance:
target_direction
1    50.288018
0    49.711982
Name: proportion, dtype: float64


# convert to stationary percentages

# 01 mvg avgs to distance percentage

In [14]:
df['sma_7_dist'] = df['close'] / df['sma_7'] - 1
df['sma_30_dist'] = df['close'] / df['sma_30'] - 1
df['sma_50_dist'] = df['close'] / df['sma_50'] - 1
df['sma_100_dist'] = df['close'] / df['sma_100'] - 1
df['sma_200_dist'] = df['close'] / df['sma_200'] - 1
df['ema_12_dist'] = df['close'] / df['ema_12'] - 1
df['ema_26_dist'] = df['close'] / df['ema_26'] - 1
df['ema_50_dist'] = df['close'] / df['ema_50'] - 1
df['ema_200_dist'] = df['close'] / df['ema_200'] - 1
df['vwap_dist'] = df['close'] / df['vwap'] - 1

# 02 macd and volatility to percentages

In [15]:
df['macd_pct'] = df['macd'] / df['close']
df['macd_sig_pct'] = df['macd_signal'] / df['close']
df['macd_hist_pct'] = df['macd_histogram'] / df['close']
df['atr_pct'] = df['atr_14'] / df['close']
df['volatility_pct'] = df['daily_volatility'] / df['close']

# drop non stationary prices

In [16]:
non_stationary_cols = [
    'open', 'high', 'low', 'close', 
    'sma_7', 'sma_30', 'sma_50', 'sma_100', 'sma_200',
    'ema_12', 'ema_26', 'ema_50', 'ema_200', 
    'vwap', 'prev_close', 'prev_high', 'prev_low',
    'macd', 'macd_signal', 'macd_histogram', 
    'atr_14', 'daily_volatility',            
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 
    'volume', 'prev_volume', 'volume_sma_20', 
    'obv' 
]

In [17]:
df = df.drop(columns=non_stationary_cols, errors='ignore')

In [18]:
df = df.dropna()
print(f"Final data shape for ml: {df.shape}")

Final data shape for ml: (3472, 31)


In [19]:
df.head(3)

,rsi_14,roc_10,roc_20,stoch_k,stoch_d,bb_percentage,volume_ratio,returns_1p,returns_5p,returns_10p,...,ema_12_dist,ema_26_dist,ema_50_dist,ema_200_dist,vwap_dist,macd_pct,macd_sig_pct,macd_hist_pct,atr_pct,volatility_pct
date,,,,,,,,,,,,,,,,,,,,,
2024-05-13 16:30:00,73.207505,1.885100,2.175111,94.345968,92.559982,1.032018,1.247968,0.001979,0.020327,0.018851,...,0.011217,0.016993,0.027924,0.066526,0.011887,0.005617,0.004401,0.001216,0.005006,0.003643
2024-05-13 17:30:00,73.117205,2.167660,2.055876,93.778076,94.439465,0.956074,0.837094,-0.000046,0.019946,0.021677,...,0.009436,0.015672,0.026755,0.065773,0.011108,0.006083,0.004737,0.001345,0.004870,0.003107
2024-05-13 18:30:00,74.472907,2.466507,2.325650,97.784559,95.302868,0.947359,0.990380,0.001821,0.007006,0.024665,...,0.009523,0.016202,0.027471,0.066994,0.012059,0.006511,0.005085,0.001426,0.004721,0.002888


# split and scale 

In [20]:
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

In [21]:
print(f"Training Data: {train_df.index.min()} to {train_df.index.max()} (Rows: {len(train_df)})")
print(f"Testing Data : {test_df.index.min()} to {test_df.index.max()} (Rows: {len(test_df)})")

Training Data: 2024-05-13 16:30:00 to 2025-12-15 20:30:00 (Rows: 2777)
Testing Data : 2025-12-16 14:30:00 to 2026-05-11 18:30:00 (Rows: 695)


# features 

In [22]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]

# scalar

In [23]:
scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

In [24]:
print(f"Scaled {len(features)} features")

Scaled 29 features


# save to parquets

In [25]:
os.makedirs('../../data/processed', exist_ok=True)
train_df.to_parquet(f'../../data/processed/train_{ASSET.lower()}_{INTERVAL}.parquet')
test_df.to_parquet(f'../../data/processed/test_{ASSET.lower()}_{INTERVAL}.parquet')
print(f"Saved: train_{ASSET.lower()}_{INTERVAL}.parquet and test_{ASSET.lower()}_{INTERVAL}.parquet")

Saved: train_aapl_1h.parquet and test_aapl_1h.parquet


In [26]:
conn.close()